# 110 — Grand Ensemble v9

**Motivation:** Grand v8 has nb104 (delta_similarity_tiers) dominating with 84% weight. Adding new OOF files (nb107-nb109) and using a 2-stage stacking approach prevents single-model dominance.

**Strategy — 2-Stage Stacking:**
- Stage 1: ElasticNet blend of all available delta-ML family OOFs → `meta_delta`
- Stage 2: ElasticNet blend of `[meta_delta, top-10 non-delta OOFs by individual RAE]`
- This prevents the delta family from collectively dominating via correlated predictions

**Alternative (if 2-stage gives worse RAE):** Run same nested-CV ElasticNet as nb96 but with ALL new OOF files (nb107-nb109 + delta_ensemble_blend).

**Both are computed and the better one is saved.**

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")

Train 4,139  Test 513  Cliffs 0


In [4]:
from sklearn.linear_model import ElasticNetCV

# --- Exclude meta-ensembles and bad models (same as nb96 logic) ---
EXCLUDE = {
    "aux_features","grand_v6","grand_v6b","grand_v6c","grand_v7",
    "grand15","grand18","grand23","grand24","grand25",
    "creative_mega_ensemble","nested_cv_ensemble",
    "cliff_role_proba","chemprop_cliff_mem_proba",
    "chemprop_chembl_nr_multitask","per_fp_stack",
    "cliff_adaptive_blend",
    # nb96 output — exclude to avoid circular blending
    "grand_v8",
}

# Delta family names (handled separately in stage 1)
DELTA_NAMES = {
    "delta_ml", "multi_template_delta", "delta_similarity_tiers",
    "delta_uncertainty", "reverse_delta_ml", "delta_5tiers",
    "delta_loso", "consensus_delta_ml", "delta_chemprop_cpu",
}

def load_oof_stack(oof_files, y_tr, exclude=None, min_std_ratio=0.4):
    """Load all valid OOF arrays; return (stack, te_stack, names)."""
    oofs, tes, names = [], [], []
    for fp in sorted(oof_files):
        name = fp.stem.replace("oof_","")
        if exclude and name in exclude: continue
        te_fp = DATA_PROCESSED / f"te_oof_{name}.npy"
        try:
            arr = np.load(fp)
            if arr.ndim > 1: arr = arr[:,0]
            if len(arr) != len(y_tr): continue
            te_v = np.load(te_fp) if te_fp.exists() else None
            if te_v is None or len(te_v) != 513: continue
            if te_v.ndim > 1: te_v = te_v[:,0]
            if te_v.std() < min_std_ratio * y_tr.std(): continue
            arr[~np.isfinite(arr)] = y_tr.mean()
            te_v[~np.isfinite(te_v)] = float(np.nanmean(te_v))
            oofs.append(arr); tes.append(te_v); names.append(name)
        except Exception as e:
            print(f"  skip {name}: {e}")
    return np.column_stack(oofs), np.column_stack(tes), names

print("Loading all OOF files...", flush=True)
all_oof_files = sorted(DATA_PROCESSED.glob("oof_*.npy"))
OOF_all, TE_all, names_all = load_oof_stack(all_oof_files, y_tr, exclude=EXCLUDE)
print(f"Total models loaded: {len(names_all)}, stack shape: {OOF_all.shape}")

# Separate delta vs non-delta
delta_idx   = [i for i,n in enumerate(names_all) if n in DELTA_NAMES]
nondelta_idx = [i for i,n in enumerate(names_all) if n not in DELTA_NAMES]
print(f"Delta models: {[names_all[i] for i in delta_idx]}")
print(f"Non-delta: {len(nondelta_idx)} models")

Loading all OOF files...


Total models loaded: 32, stack shape: (4139, 32)
Delta models: ['delta_5tiers', 'delta_chemprop_cpu', 'delta_loso', 'delta_ml', 'delta_similarity_tiers', 'delta_uncertainty', 'multi_template_delta', 'reverse_delta_ml']
Non-delta: 24 models


In [5]:
# --- Compute individual RAEs for model selection ---
individual_raes = {n: rae(y_tr, OOF_all[:,i]) for i,n in enumerate(names_all)}
rae_df = pd.DataFrame([(n,r) for n,r in individual_raes.items()], columns=["model","RAE"]).sort_values("RAE")
print("Top-15 models by individual OOF RAE:")
print(rae_df.head(15).round(4).to_string(index=False))

# Top-10 non-delta models by RAE
nondelta_names_sorted = [(n, individual_raes[n]) for n in names_all if n not in DELTA_NAMES]
nondelta_names_sorted.sort(key=lambda x: x[1])
top10_nondelta = [n for n,_ in nondelta_names_sorted[:10]]
print(f"\nTop-10 non-delta for Stage 2: {top10_nondelta}")

Top-15 models by individual OOF RAE:
                 model    RAE
  delta_ensemble_blend 0.2748
delta_similarity_tiers 0.2772
          delta_5tiers 0.2888
            delta_loso 0.3266
  multi_template_delta 0.3266
     delta_uncertainty 0.3268
      reverse_delta_ml 0.3269
              delta_ml 0.4164
   stochastic_ensemble 0.5550
            smiles_aug 0.5581
    delta_chemprop_cpu 0.5585
     multi_nr_transfer 0.5609
     pxr_pharmacophore 0.5611
    3d_shape_conformer 0.5619
    bio_nr_fingerprint 0.5621

Top-10 non-delta for Stage 2: ['delta_ensemble_blend', 'stochastic_ensemble', 'smiles_aug', 'multi_nr_transfer', 'pxr_pharmacophore', '3d_shape_conformer', 'bio_nr_fingerprint', 'free_wilson', 'tox21_bio_fp', '3d_shape']


In [6]:
# ==========================================================
# APPROACH A: 2-Stage Stacking
# Stage 1: ElasticNet blend of delta family -> meta_delta
# Stage 2: ElasticNet blend of [meta_delta, top-10 non-delta]
# ==========================================================
print("\n=== Approach A: 2-Stage Stacking ===", flush=True)

if len(delta_idx) < 2:
    print("  Not enough delta models, skipping 2-stage approach")
    oof_2stage = None
    te_2stage = None
else:
    OOF_delta  = OOF_all[:, delta_idx]
    TE_delta   = TE_all[:, delta_idx]
    delta_names = [names_all[i] for i in delta_idx]

    # Get indices of top-10 non-delta
    top10_nd_idx = [names_all.index(n) for n in top10_nondelta]
    OOF_top10  = OOF_all[:, top10_nd_idx]
    TE_top10   = TE_all[:, top10_nd_idx]

    oof_s1   = np.full(len(y_tr), np.nan)  # stage 1 meta_delta
    oof_2stage = np.full(len(y_tr), np.nan)  # final 2-stage

    for k, (tr_idx, va_idx) in enumerate(splits):
        meta_tr_idx = [i for fold,(ti,_) in enumerate(splits) for i in ti if fold!=k]

        # Stage 1: blend delta family
        s1 = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
        s1.fit(OOF_delta[meta_tr_idx], y_tr[meta_tr_idx])
        oof_s1[va_idx] = s1.predict(OOF_delta[va_idx])

        # Stage 2: blend [meta_delta, top-10 non-delta]
        # Rebuild stage 2 input for all meta_tr_idx using stage1
        s1_meta_tr = s1.predict(OOF_delta[meta_tr_idx])
        s2_tr_input = np.column_stack([s1_meta_tr, OOF_top10[meta_tr_idx]])
        s2_va_input = np.column_stack([oof_s1[va_idx], OOF_top10[va_idx]])

        s2 = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
        s2.fit(s2_tr_input, y_tr[meta_tr_idx])
        oof_2stage[va_idx] = s2.predict(s2_va_input)

        print(f"  fold {k+1}  s1_RAE={rae(y_tr[va_idx], oof_s1[va_idx]):.4f}  "
              f"s2_RAE={rae(y_tr[va_idx], oof_2stage[va_idx]):.4f}", flush=True)

    m_s1 = full_metrics(y_tr, oof_s1,     cliff_pairs, "stage1_delta_blend")
    m_2s = full_metrics(y_tr, oof_2stage,  cliff_pairs, "2stage_v9")
    print(f"\n2-Stage OOF RAE: {m_2s['RAE']:.4f}")

    # Final test predictions (2-stage)
    s1_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
    s1_final.fit(OOF_delta, y_tr)
    te_s1 = s1_final.predict(TE_delta)

    s2_te_input = np.column_stack([te_s1, TE_top10])
    s2_tr_all   = np.column_stack([s1_final.predict(OOF_delta), OOF_top10])
    s2_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
    s2_final.fit(s2_tr_all, y_tr)
    te_2stage = np.clip(s2_final.predict(s2_te_input), y_tr.min()-0.5, y_tr.max()+0.5)
    print(f"2-stage weights: {dict(zip(['meta_delta']+top10_nondelta, s2_final.coef_.round(4).tolist()))}")


=== Approach A: 2-Stage Stacking ===


  fold 1  s1_RAE=0.2550  s2_RAE=0.2523


  fold 2  s1_RAE=0.2678  s2_RAE=0.2644


  fold 3  s1_RAE=0.2966  s2_RAE=0.2951


  fold 4  s1_RAE=0.2842  s2_RAE=0.2842


  fold 5  s1_RAE=0.2901  s2_RAE=0.2889


  [stage1_delta_blend] RAE=0.2766 MAE=0.2517 R2=0.8536 r=0.9239 rho=0.8993 tau=0.7555
  [2stage_v9] RAE=0.2748 MAE=0.2500 R2=0.8548 r=0.9245 rho=0.9001 tau=0.7567

2-Stage OOF RAE: 0.2748


2-stage weights: {'meta_delta': 0.0, 'delta_ensemble_blend': 0.9936, 'stochastic_ensemble': 0.0, 'smiles_aug': 0.0, 'multi_nr_transfer': -0.0, 'pxr_pharmacophore': 0.0048, '3d_shape_conformer': 0.0795, 'bio_nr_fingerprint': 0.0484, 'free_wilson': 0.0, 'tox21_bio_fp': 0.0029, '3d_shape': -0.126}


In [7]:
# ==========================================================
# APPROACH B: Flat nested-CV ElasticNet over ALL OOF files
# Same as nb96 but with new OOF files added
# ==========================================================
print("\n=== Approach B: Flat nested-CV ElasticNet over all models ===", flush=True)
oof_flat = np.full(len(y_tr), np.nan)

for k, (tr_idx, va_idx) in enumerate(splits):
    meta_tr_idx = [i for fold,(ti,_) in enumerate(splits) for i in ti if fold!=k]
    meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
    meta.fit(OOF_all[meta_tr_idx], y_tr[meta_tr_idx])
    oof_flat[va_idx] = meta.predict(OOF_all[va_idx])
    print(f"  fold {k+1}  val_RAE={rae(y_tr[va_idx], oof_flat[va_idx]):.4f}", flush=True)

m_flat = full_metrics(y_tr, oof_flat, cliff_pairs, "flat_v9")
print(f"\nFlat v9 OOF RAE: {m_flat['RAE']:.4f}")

# Final test predictions (flat)
meta_flat_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
meta_flat_final.fit(OOF_all, y_tr)
te_flat = np.clip(meta_flat_final.predict(TE_all), y_tr.min()-0.5, y_tr.max()+0.5)

coef_flat = pd.DataFrame({"model": names_all, "weight": meta_flat_final.coef_}).sort_values("weight", ascending=False)
print("\nTop weights (flat v9):")
print(coef_flat[coef_flat.weight.abs()>1e-6].head(15).to_string(index=False))


=== Approach B: Flat nested-CV ElasticNet over all models ===


  fold 1  val_RAE=0.2512


  fold 2  val_RAE=0.2664


  fold 3  val_RAE=0.2962


  fold 4  val_RAE=0.2857


  fold 5  val_RAE=0.2899


  [flat_v9] RAE=0.2757 MAE=0.2508 R2=0.8563 r=0.9253 rho=0.9009 tau=0.7572

Flat v9 OOF RAE: 0.2757

Top weights (flat v9):
               model   weight
delta_ensemble_blend 0.978681
   multi_fp_ensemble 0.013022


In [8]:
# --- Also load nb96 (grand_v8) for comparison ---
v8_path = DATA_PROCESSED / "oof_grand_v8.npy"
if v8_path.exists():
    oof_v8 = np.load(v8_path)
    m_v8 = full_metrics(y_tr, oof_v8, cliff_pairs, "grand_v8")
    print(f"Grand v8 OOF RAE: {m_v8['RAE']:.4f}")

# --- Select best approach ---
candidates = [("flat_v9", m_flat["RAE"], oof_flat, te_flat)]
if oof_2stage is not None:
    candidates.append(("2stage_v9", m_2s["RAE"], oof_2stage, te_2stage))
candidates.sort(key=lambda x: x[1])
best_name, best_rae_v, oof, te_preds = candidates[0]

print(f"\n=== Grand v9 Selection ===")
for name, r, _, _ in candidates:
    marker = " <-- BEST" if name == best_name else ""
    print(f"  {name}: OOF RAE = {r:.4f}{marker}")

m_final_v9 = full_metrics(y_tr, oof, cliff_pairs, f"grand_v9_{best_name}")
m_final_v9_a = full_metrics(y_tr[active_mask], oof[active_mask], label="v9 [active]")
print(f"\nGrand v9 OOF RAE: {m_final_v9['RAE']:.4f}")
print(pd.DataFrame([m_final_v9, m_final_v9_a],
                    index=["overall","active"]).round(4).to_string())

  [grand_v8] RAE=0.2843 MAE=0.2586 R2=0.8402 r=0.9166 rho=0.8889 tau=0.7443
Grand v8 OOF RAE: 0.2843

=== Grand v9 Selection ===
  2stage_v9: OOF RAE = 0.2748 <-- BEST
  flat_v9: OOF RAE = 0.2757
  [grand_v9_2stage_v9] RAE=0.2748 MAE=0.2500 R2=0.8548 r=0.9245 rho=0.9001 tau=0.7567
  [v9 [active]] RAE=2.0286 MAE=0.4254 R2=-3.5793 r=0.2664 rho=0.2041 tau=0.1553

Grand v9 OOF RAE: 0.2748
            RAE     MAE      R2  Pearson  Spearman  Kendall
overall  0.2748  0.2500  0.8548   0.9245    0.9001   0.7567
active   2.0286  0.4254 -3.5793   0.2664    0.2041   0.1553


In [9]:
# --- Save ---
np.save(DATA_PROCESSED/"oof_grand_v9.npy", oof)
np.save(DATA_PROCESSED/"te_oof_grand_v9.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"110_grand_ensemble_v9.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")

# Compare v8 vs v9
if v8_path.exists():
    print(f"\n=== v8 vs v9 ===")
    print(f"  Grand v8 OOF RAE: {m_v8['RAE']:.4f}")
    print(f"  Grand v9 OOF RAE: {m_final_v9['RAE']:.4f}")
    delta_rae = m_final_v9["RAE"] - m_v8["RAE"]
    direction = "improvement" if delta_rae < 0 else "regression"
    print(f"  Delta: {delta_rae:+.4f} ({direction})")

print(f"\n*** Grand v9 OOF RAE = {m_final_v9['RAE']:.4f} (approach: {best_name}) ***")

Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\110_grand_ensemble_v9.csv
Test: min=3.06 med=4.97 max=6.63

=== v8 vs v9 ===
  Grand v8 OOF RAE: 0.2843
  Grand v9 OOF RAE: 0.2748
  Delta: -0.0094 (improvement)

*** Grand v9 OOF RAE = 0.2748 (approach: 2stage_v9) ***
